# Dự án 4: SQL BigQuery case study (tài liệu hóa)

## Dự án này là gì?
Notebook này tổng hợp lại phần làm bài SQL trong file `document_task9.xlsx` để bạn có thể
đưa vào portfolio như một **case study phân tích dữ liệu bằng BigQuery**.

## Lưu ý quan trọng
Hiện tại **chưa có bảng nguồn** nên notebook này không hiển thị output thực tế.
Các bảng còn thiếu gồm:
- `events`
- `user_info`
- `geography`

Vì vậy đây là **notebook tài liệu hóa giải pháp**: bạn có thể dùng ngay để chứng minh tư duy SQL,
còn nếu sau này có raw data/BigQuery access thì chỉ cần chạy lại các truy vấn.

## 1. Thống kê mô tả

        Mục tiêu:
        - số user sử dụng mỗi ngày
        - số lượt access mỗi ngày
        - phân tích theo platform
        - khối lượng volume trung bình theo thời gian
        - số user theo quốc gia

        ```sql
        SELECT
  e.date,
  COUNT(DISTINCT e.user_id) AS daily_user,
  COUNT(*) AS total_access
FROM `ace-slice-459110-f8.test9.events` e
GROUP BY e.date
ORDER BY e.date;
        ```

        ```sql
        SELECT
  e.date,
  e.platform,
  COUNT(DISTINCT user_id) AS daily_user,
  COUNT(*) AS total_access
FROM `ace-slice-459110-f8.test9.events` e
GROUP BY e.date, e.platform
ORDER BY e.date ASC;
        ```

        ```sql
        SELECT
  AVG(e.volume) AS avg_vol,
  e.date
FROM `ace-slice-459110-f8.test9.events` e
GROUP BY e.date
ORDER BY e.date ASC;
        ```

        ```sql
        SELECT
  g.Country,
  COUNT(DISTINCT u.user_id) AS total_users
FROM `ace-slice-459110-f8.test9.user_info` u
JOIN `ace-slice-459110-f8.test9.geography` g
  ON u.mp_country_code = g.Code
GROUP BY g.Country
ORDER BY total_users DESC;
        ```

## 2. Dashboard / trực quan hóa

Yêu cầu trong brief:
- tạo dashboard trên Looker Studio
- có filter theo `platform`
- có filter theo `country`
- highlight insight quan trọng cho sếp

### Gợi ý bố cục dashboard
- KPI card: Daily users, total access, avg volume
- Line chart: daily users theo ngày
- Stacked bar/line: access theo platform
- Map hoặc bar chart: user theo quốc gia
- Insight box: biến động nổi bật / top quốc gia / top platform

## 3. Retention rate

        Ý tưởng:
        - tìm ngày đầu tiên user sử dụng app (`day 0`)
        - lấy danh sách các ngày user quay lại
        - tính khoảng cách ngày so với `day 0`
        - đếm retained users theo từng `day_diff`
        - chia cho số user ngày 0 để ra retention rate

        ```sql
        WITH fi AS (
  SELECT
    e.user_id,
    MIN(e.date) AS first_use_date
  FROM `ace-slice-459110-f8.test9.events` e
  GROUP BY e.user_id
),
ba AS (
  SELECT DISTINCT
    e.user_id,
    e.date
  FROM `ace-slice-459110-f8.test9.events` e
  ORDER BY e.user_id ASC
),
day_dif AS (
  SELECT
    f.user_id,
    DATE_DIFF(b.date, f.first_use_date, DAY) AS day_diff
  FROM fi f
  JOIN ba b
    ON f.user_id = b.user_id
),
retained AS (
  SELECT
    d.day_diff,
    COUNT(DISTINCT d.user_id) AS retained_users
  FROM day_dif d
  GROUP BY d.day_diff
  ORDER BY d.day_diff ASC
)
SELECT
  r.day_diff,
  r.retained_users,
  FIRST_VALUE(retained_users) OVER (ORDER BY day_diff ASC) AS users_day0,
  ROUND((retained_users * 100.0) / FIRST_VALUE(retained_users) OVER (ORDER BY day_diff ASC), 2) AS retention_rate_pct
FROM retained r;
        ```

## 4. Phát hiện gian lận / lạm dụng trial

        Logic phân tích trong workbook:
        - **Layer 1:** nhóm tài khoản có trùng cụm thông tin (`country`, `city`, `os`, `platform`)
        - **Layer 2:** tổng hợp hành vi sử dụng (`first_event_day`, `last_event_day`, `total_vol`, `total_fee`)
        - **Sequencing:** so sánh account mới với account trước đó trong cùng cụm đặc điểm
        - Tính khoảng cách giữa `created_date` mới và `last_event_day` cũ để tìm trường hợp đáng nghi

        ```sql
        WITH layer1 AS (
  SELECT DISTINCT
    user_id,
    mp_country_code,
    city,
    os,
    platform,
    created_date
  FROM (
    SELECT
      *,
      COUNT(*) OVER (
        PARTITION BY mp_country_code, city, os, platform
      ) AS so_trong_nhom
    FROM `ace-slice-459110-f8.test9.user_info`
    WHERE mp_country_code IS NOT NULL
      AND city IS NOT NULL
      AND os IS NOT NULL
      AND platform IS NOT NULL
      AND user_id IS NOT NULL
      AND created_date IS NOT NULL
  )
  WHERE so_trong_nhom > 1
),
layer2 AS (
  SELECT
    e.user_id,
    MIN(e.date) AS first_event_day,
    MAX(e.date) AS last_event_day,
    SUM(e.volume) AS total_vol,
    SUM(e.fee) AS total_fee,
    COUNT(*) AS total_event
  FROM `ace-slice-459110-f8.test9.events` e
  WHERE e.user_id IS NOT NULL
  GROUP BY e.user_id
),
joined AS (
  SELECT
    l1.user_id,
    l1.mp_country_code,
    l1.city,
    l1.os,
    l1.platform,
    l1.created_date,
    l2.first_event_day,
    l2.last_event_day,
    l2.total_vol,
    l2.total_fee,
    l2.total_event
  FROM layer1 l1
  JOIN layer2 l2
    ON l1.user_id = l2.user_id
),
sequenced AS (
  SELECT
    *,
    LAG(user_id) OVER (
      PARTITION BY mp_country_code, city, os, platform
      ORDER BY created_date
    ) AS prev_user_id,
    LAG(last_event_day) OVER (
      PARTITION BY mp_country_code, city, os, platform
      ORDER BY created_date
    ) AS prev_last
  FROM joined
)
SELECT
  *,
  DATE_DIFF(created_date, prev_last, DAY) AS gap_ngay_tao,
  DATE_DIFF(first_event_day, prev_last, DAY) AS gap_ngay_su_dung
FROM sequenced
WHERE prev_user_id IS NOT NULL
ORDER BY mp_country_code, city, os, platform, created_date;
        ```

## 5. Tính lại số lượt truy cập (access recalculation)

        Business rule mới:
        - nếu user quay lại trong vòng **6 giờ** thì vẫn tính là **1 access**
        - volume vẫn cộng bình thường

        ```sql
        WITH ordered AS (
  SELECT
    e.user_id,
    e.datetime,
    e.volume,
    LAG(e.datetime) OVER (PARTITION BY e.user_id ORDER BY e.datetime) AS prev_time
  FROM `ace-slice-459110-f8.test9.events` e
  WHERE e.user_id IS NOT NULL
),
accesses AS (
  SELECT
    *,
    TIMESTAMP_DIFF(o.datetime, prev_time, HOUR) AS gap_hours,
    CASE
      WHEN prev_time IS NULL THEN 1
      WHEN TIMESTAMP_DIFF(o.datetime, prev_time, HOUR) > 6 THEN 1
      ELSE 0
    END AS is_new_access
  FROM ordered o
  ORDER BY user_id, datetime
)
SELECT
  a.user_id,
  SUM(a.is_new_access) AS total_access_new,
  COUNT(*) AS total_access_old,
  COUNT(*) - SUM(a.is_new_access) AS chenh_lech,
  SUM(a.volume) AS total_volume
FROM accesses a
GROUP BY a.user_id;
        ```

## 6. Customer segmentation bằng RFM

        Tư duy chính:
        - `Recency`: bao lâu rồi user chưa quay lại
        - `Frequency`: số access sau khi chuẩn hóa rule 6 giờ
        - `Monetary`: tổng volume
        - dùng `NTILE(5)` để chấm điểm R, F, M
        - gom thành các nhãn: `Yeu`, `Trung_binh`, `Tiem_nang`, `VIP`

        ```sql
        WITH ordered AS (
  SELECT
    e.user_id,
    e.date,
    e.datetime,
    e.volume,
    LAG(e.datetime) OVER (
      PARTITION BY e.user_id
      ORDER BY e.datetime
    ) AS prev_time
  FROM `ace-slice-459110-f8.test9.events` e
  WHERE e.user_id IS NOT NULL
),
accesses AS (
  SELECT
    *,
    TIMESTAMP_DIFF(datetime, prev_time, MINUTE) AS gap_minutes,
    CASE
      WHEN prev_time IS NULL THEN 1
      WHEN TIMESTAMP_DIFF(datetime, prev_time, MINUTE) > 360 THEN 1
      ELSE 0
    END AS is_new_access
  FROM ordered
),
max_day AS (
  SELECT MAX(e.date) AS dataset_max_date
  FROM `ace-slice-459110-f8.test9.events` e
),
RFM AS (
  SELECT
    a.user_id,
    SUM(a.is_new_access) AS Frequency,
    SUM(a.volume) AS Monetary,
    DATE_DIFF(m.dataset_max_date, MAX(a.date), DAY) AS recency_days
  FROM accesses a
  CROSS JOIN max_day m
  GROUP BY a.user_id, m.dataset_max_date
),
score AS (
  SELECT
    *,
    NTILE(5) OVER (ORDER BY Frequency ASC) AS F_score,
    NTILE(5) OVER (ORDER BY Monetary ASC) AS M_score,
    NTILE(5) OVER (ORDER BY recency_days DESC) AS R_score
  FROM RFM
)
SELECT
  *,
  (F_score + M_score + R_score) AS RFM_score,
  CASE
    WHEN (F_score + M_score + R_score) < 6 THEN 'Yeu'
    WHEN (F_score + M_score + R_score) < 9 THEN 'Trung_binh'
    WHEN (F_score + M_score + R_score) < 12 THEN 'Tiem_nang'
    ELSE 'VIP'
  END AS Phan_loai
FROM score;
        ```

## 7. Cách đưa case study này vào portfolio/CV

Bạn có thể mô tả ngắn như sau:

- Xây dựng truy vấn BigQuery để đo daily users, total access và volume theo thời gian.
- Thiết kế logic retention rate theo cohort day 0 và tính retention cho các mốc ngày tiếp theo.
- Phân tích dấu hiệu gian lận trial bằng window functions, sequence logic và hành vi tạo account mới.
- Chuẩn hóa business rule access (session gộp trong 6 giờ) để báo cáo phản ánh đúng hành vi người dùng.
- Thực hiện customer segmentation bằng mô hình RFM và phân nhóm người dùng theo mức độ giá trị.